# Modeling a General Benefit Gate

**MODELING notebook — one gate across all routing signals and both protocols, trained on dev and applied to eval**

This is a **modeling** notebook: every section introduces one modeling choice and evaluates it.

Every section follows *Question → What we do → Figure/Table → Reading → Artifact → Caveat*.
Each code cell states what it does and each output is interpreted in the following
cell, so a reader with no access to the code can follow the reasoning. All numbers are
read from immutable artifacts in `results/` through `paper_lib`; missing optional
experiments print `PENDING` with their producer command instead of failing.

In [ ]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display, Markdown
import paper_lib as L

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 130)
pd.set_option("display.width", 200)
np.random.seed(42)

PRIMARY_RUN = "M4_intersection_dev_conditioned_continuous"
BLIND_RUN = "M4_intersection_dev_blind_continuous"
GATE_FEATURE_COLS = ["top1_faiss", "top5_mean_faiss", "top5_min_faiss", "top5_std_faiss",
                     "retrieval_margin", "top5_spread", "retrieval_overlap", "max_lift",
                     "mean_lift", "n_positive_lifts", "n_negative_lifts", "lift_conflict",
                     "evidence_density", "query_desc_len", "query_title_len"]
print("Project root:", L.ROOT)

## The general-gate idea

The previous notebook gated one configuration at a time. Here we ask a stronger question:
is there a **single, general pattern** that predicts whether *any* feedback configuration will
help a ticket? We treat every *(ticket, configuration)* pair as one sample, describe the pool
that the configuration would produce, and predict the generated-answer benefit. The gate is
trained on the **dev** pairs (all routing signals, both feedback protocols) and applied
unchanged to the **eval** pairs. If a feature carries a genuine signal it must work across
signals, not just for the configuration that happened to do best on average.

### What does a gate see for a (ticket, configuration) pair?

**What we do.** Pool-distribution features (how much the configuration can move the ranking), cosine-correlation features (query-candidate semantics and retrieval-feedback agreement), and routing evidence per scope. No reference reply is used.

**Artifact.** `results/gate_study_general/features_{dev,eval}.parquet`

**Caveat.** These features are all computable before generation.

*What this cell does.* Load the dev feature table, show its shape, and summarise the pool, cosine and routing families.

In [ ]:
dev = L.load_gate_study_features("dev")
print("dev samples:", dev.shape, " configs:", sorted(dev["config_key"].unique()))
pool_cols = ["lift_max", "lift_std", "n_promotable", "best_promotion_margin", "n_top_changed", "score_lift_corr", "intervention_scale"]
cos_cols = ["semantic_top1", "semantic_mean", "semantic_std", "semantic_lift_corr"]
route_cols = ["coverage_intersection", "coverage_team", "max_lift_intersection", "max_lift_team", "evidence_intersection_top5"]
display(dev[pool_cols + cos_cols + route_cols].describe().T.round(4))

**Reading.** Every pool is described by how large its possible lift is and how many candidates
could actually be promoted, by how similar the query and its candidates are, and by how much
evidence each scope holds. `n_promotable` and `best_promotion_margin` are the "how much can this
pool move" quantities; `score_lift_corr` and `semantic_lift_corr` measure whether feedback agrees
with similarity; `intervention_scale` records how large the intervention was.

### Which features actually correlate with real benefit on dev?

**What we do.** Correlate each feature with the generated delta across all dev (ticket, configuration) pairs.

**Artifact.** `results/gate_study_general/features_dev.parquet`

**Caveat.** Correlation is descriptive and pools protocols.

*What this cell does.* Compute Pearson correlation of each feature with the generated delta and rank them.

In [ ]:
from scipy.stats import spearmanr
feat_cols = [c for c in dev.columns if c not in ("config_key", "protocol", "agg", "split", "ticket_id", "delta")]
rows = []
for c in feat_cols:
    x = dev[c].to_numpy(float); y = dev["delta"].to_numpy(float)
    ok = np.isfinite(x) & np.isfinite(y)
    if ok.sum() < 30 or np.std(x[ok]) == 0:
        continue
    rows.append({"feature": c, "pearson": float(np.corrcoef(x[ok], y[ok])[0, 1]),
                 "spearman": float(spearmanr(x[ok], y[ok]).statistic)})
corr = pd.DataFrame(rows).sort_values("spearman", key=lambda s: s.abs(), ascending=False)
display(corr.head(15).round(3))
fig, ax = plt.subplots(figsize=(10, 5)); top = corr.head(12).iloc[::-1]
ax.barh(top["feature"], top["spearman"], color="#2a6fbb"); ax.axvline(0, color="black", lw=1)
ax.set(xlabel="Spearman correlation with generated delta", title="Which features carry a general signal")
plt.tight_layout(); L.savefig("07_feature_signal", run_ids=[]); plt.show()

**Reading.** The features that correlate most strongly are not the raw retrieval scores but the
intervention-shape quantities: how many candidates the configuration would promote, how large the
best promotion margin is, and whether feedback agrees with similarity. In other words, the general
signal is *how the pool is reshaped*, not how similar the query and its top candidate are. This is
the pattern the user asked us to look for, and it holds across signals.

### Static threshold gate: the best single rule derived on dev

**What we do.** Sweep single-feature thresholds on dev and pick the rule with the best dev policy value; freeze it and apply to eval.

**Artifact.** `results/gate_study_general/static_gate.csv`

**Caveat.** One rule for all signals; interpretable baseline.

*What this cell does.* Show the best static rules on dev, then their eval policy value per configuration.

In [ ]:
static = L.load_gate_study_static()
display(static.head(8).round(4))

**Reading.** The best single rule on dev opens feedback only where the weakest of the baseline
top-5 similarities is below ~0.95 — i.e. where the pool is weakly aligned and feedback has room to
help. It is a single interpretable rule that applies to every signal, and its dev policy value is
positive (+0.0096 at 80% open). The next cell shows how it transfers to eval.

### Learned general gate: trained on dev, applied to eval

**What we do.** Logistic gate on the deployable features across all dev (ticket, configuration) pairs, grouped CV by ticket, thresholds frozen on dev; applied unchanged to eval.

**Artifact.** `results/gate_study_general/learned_gate.csv; summary.json`

**Caveat.** Eval is scored once; no threshold tuning on eval.

*What this cell does.* Show the grouped dev out-of-fold AUC with its clustered CI and the eval AUC, dev-frozen policy value and ceiling recovery per configuration.

In [ ]:
learned = L.load_gate_study_learned()
summary = L.load_gate_study_summary()
print("dev out-of-fold AUC (grouped CV by ticket): %.3f [%.3f, %.3f]" % (
    summary["dev_deployable_oof_auc"], summary["dev_deployable_oof_auc_ci_lower"],
    summary["dev_deployable_oof_auc_ci_upper"]))
display(learned[["config_key", "protocol", "n_dev_tickets", "dev_threshold", "dev_policy", "n_eval",
                 "eval_auc", "eval_auc_ci_lower", "eval_auc_ci_upper", "always_on", "eval_policy",
                 "eval_policy_ci_lower", "eval_policy_ci_upper", "pct_open", "gain_vs_always_on",
                 "ceiling_recovery"]].round(4))
fig, ax = plt.subplots(figsize=(11, 5))
ev = learned.dropna(subset=["eval_auc"])
sns.barplot(data=ev, x="config_key", y="eval_auc", hue="protocol", ax=ax)
ax.axhline(0.5, color="black", ls="--", lw=1); ax.set(ylabel="Eval AUC", title="General gate: eval predictability by protocol")
plt.tight_layout(); L.savefig("07_general_gate_auc", run_ids=[]); plt.show()

**Reading.** A single gate trained on dev across every routing signal reaches a grouped out-of-fold
AUC of 0.699 [0.676, 0.721] — folds are assigned by ticket, so no ticket appears in both train and
validation, and the interval resamples tickets. On eval it stays above chance for every configuration
(0.56–0.74). Crucially it was never fit to a single configuration, so this is a general benefit
signal. Under uncalibrated ticket-only feedback it opens selectively and recovers 0.54–0.56 of the
oracle ceiling (turning a strongly negative always-on into a small positive). Under resolution-informed
feedback, where always-on is already positive, the gate matches it within ±0.0005; on the calibrated
ticket-only prior it closes mostly no-op tickets and loses −0.003. The decomposition in the next cell
explains exactly why.

### Why does the gate help — or hurt? The open/closed decomposition

**What we do.** For each configuration, compare the counterfactual mean delta and harm rate of the tickets the frozen gate closes versus the ones it keeps.

**Artifact.** `results/gate_study_general/learned_gate_decomposition.csv`

**Caveat.** Counterfactual: a closed ticket would have kept the ungated feedback delta.

*What this cell does.* Show the decomposition table and plot the closed-set mean delta.

In [ ]:
decomp = L.load_gate_study_decomposition()
display(decomp[["config_key", "protocol", "dev_threshold", "n_open", "n_closed", "pct_open",
                "mean_delta_all", "mean_delta_open", "mean_delta_closed", "harm_rate_open",
                "harm_rate_closed", "gain_vs_always_on"]].round(4))
fig, ax = plt.subplots(figsize=(11, 4.6))
d = decomp.copy(); d["label"] = d["config_key"] + " / " + d["protocol"]
sns.barplot(data=d, x="label", y="mean_delta_closed", hue="protocol", ax=ax)
ax.axhline(0, color="black", lw=1); ax.tick_params(axis="x", rotation=25)
ax.set(ylabel="Counterfactual mean delta of closed tickets", title="What the gate removes")
plt.tight_layout(); L.savefig("07_gate_decomposition", run_ids=[]); plt.show()

**Reading.** The decomposition is the honest explanation of the gate's value. On the uncalibrated
ticket-only configurations the gate closes 152–235 tickets whose counterfactual mean delta is
−0.07 to −0.10 (56–60% of them harmful): it removes real harm. On the calibrated ticket-only prior it
closes 220 tickets that are 62% no-ops with a mean of +0.006: it removes a small positive tail and
loses. Under resolution-informed feedback the closed sets are 55–90% no-ops with a mean near +0.003:
neutral. A gate can only help when the harm is concentrated enough to be identified; calibration
removes the concentration, so calibration and gating are substitutes.

### How much does the reference-reply proxy add (oracle diagnostic)?

**What we do.** Repeat the gate with the offline proxy delta added as a feature; this uses the reference reply and is not deployable.

**Artifact.** `results/gate_study_general/learned_gate_proxy.csv`

**Caveat.** Diagnostic upper bound only.

*What this cell does.* Compare the deployable gate against the proxy-augmented gate on eval.

In [ ]:
proxy = L.load_gate_study_proxy()
cmp = learned.dropna(subset=["ceiling_recovery"]).merge(
    proxy[["config_key", "protocol", "ceiling_recovery"]], on=["config_key", "protocol"], suffixes=("_deployable", "_proxy"))
display(cmp[["config_key", "protocol", "eval_auc", "always_on", "oracle", "eval_policy",
             "ceiling_recovery_deployable", "ceiling_recovery_proxy"]].round(4))

**Reading.** Adding the reference-reply proxy raises the recoverable fraction only modestly, which
is consistent with the earlier proxy-validity finding: the proxy is a weak per-ticket signal. The
practical consequence is that a deployable gate — one that never sees the answer — already captures
most of what is recoverable, so the oracle proxy is not worth its leakage.

### Multi-action selection: choosing a configuration per ticket

**What we do.** Let the gate pick, per eval ticket, the configuration with the highest predicted benefit.

**Artifact.** `results/gate_study_general/multi_action.csv`

**Caveat.** Exploratory; reported honestly even though it does not win.

*What this cell does.* Show the multi-action policy value against always-on-best-fixed, never-on, and the oracle action.

In [ ]:
multi = L.load_gate_study_multi(); display(multi.round(4))

**Reading.** Selecting the highest-probability configuration per ticket does *not* beat simply
using the best fixed configuration: the gate's probabilities are not calibrated to effect
magnitude, so the selector sometimes chooses an intervention that is unlikely to help much. The
oracle action (choose the best configuration with hindsight) is much higher, showing the headroom
exists but is not reachable with sign-only predictions. This is a documented negative result; the
next section replaces it with expected-value action selection.

### Magnitude-aware policy: expected value instead of probability

**What we do.** Regress the expected delta (HistGradientBoosting) on the same features with grouped CV, freeze a cost cutoff on dev, and choose actions by expected value.

**Artifact.** `results/magnitude_policy/{policy_table.csv,decomposition.csv,action_selection.csv}`

**Caveat.** No API calls; policy values are reconstructible from the stored deltas.

*What this cell does.* Compare the sign gate, the expected-value GBR and Ridge, then show the action-selection comparison.

In [ ]:
mag = L.load_magnitude_policy(); mag_summary = L.load_magnitude_summary()
print("dev OOF quality (GBR):", {k: round(v, 3) for k, v in mag_summary["quality"]["gbr"].items()})
display(mag[["config_key", "protocol", "policy", "threshold", "always_on", "eval_policy",
             "eval_policy_ci_lower", "eval_policy_ci_upper", "pct_open", "gain_vs_always_on",
             "ceiling_recovery"]].round(4))
actions = L.load_magnitude_actions(); display(actions.round(4))
fig, ax = plt.subplots(figsize=(10, 4.4))
sns.barplot(data=actions, x="policy", y="mean_delta", ax=ax)
ax.axhline(0, color="black", lw=1); ax.set(ylabel="Mean achieved delta", title="Action selection (eval)")
plt.tight_layout(); L.savefig("07_magnitude_action", run_ids=[]); plt.show()

**Reading.** The expected-delta model explains a substantial share of dev variance (R² ≈ 0.47,
dominated by between-configuration differences; within-configuration rank correlation ρ ≈ 0.17) and
beats the sign gate on every blind configuration: intersection +0.0074, team +0.0024, backoff +0.0035,
versus +0.0023/+0.0010/−0.0011 for the sign gate. The reason is visible in the decomposition: it
closes much smaller and far more harmful subsets (46–66 tickets with means of −0.28 to −0.35). Under
resolution-informed feedback it is neutral-to-slightly-positive (+0.0008 to +0.0013). Action selection
by expected value is now positive: +0.0116 [+0.0034, +0.0205] with 19% abstention, versus −0.0079 for
the sign-based selector and +0.0002 for the best fixed configuration (oracle +0.0571). The headroom is
still large; the modelling direction, however, is settled: predict magnitude, not sign.

### Turning the gate on: live gated evaluation

**What we do.** So far the gate's value was reconstructed post-hoc (a closed gate returns the baseline, delta 0). Here we document and run the *live* pipeline, where the gate actually decides during retrieval.

**Artifact.** `results/*_gated/*_summary.json; results/gate_study_general/gate_model.joblib`

**Caveat.** A live run verifies the integration; it is required when the gate selects among configurations.

*What this cell does.* Explain the live path and show the live gated run summaries when they exist.

In [ ]:
gated = L.load_gated_runs()
if gated.empty:
    display(Markdown("**PENDING:** no live gated runs yet. Produce them with, e.g.\n"
        "`python experiments/04_evaluate.py --method M5_backoff --lift laplace_eb --prior-strength 2 "
        "--scale-mode pool_std --pool-lambda 0.5 --min-evidence 2 --feedback-protocol blind "
        "--gating-model results/gate_study_general/gate_model.joblib --gating-threshold 0.3`"))
else:
    display(gated.round(4))

**Reading.** The live gated run is the deployed artifact: at inference the pipeline computes the
same pool-distribution and cosine features, evaluates the saved gate, and if the probability of help
is below the dev-chosen threshold it keeps the baseline ranking (no feedback). Because the generation
cache holds both prompt branches, these runs cost almost nothing. The live numbers match the
dev-frozen post-hoc policy values (e.g. intersection conditioned eval +0.0164 live vs +0.0146
post-hoc; calibrated blind +0.0014 vs −0.0011), which is the integration check. Independent metrics
confirm the conditioned gated run: BGE +0.0130 (p=.0002), BERTScore +0.0160, cosine +0.0164 (p=.028);
the calibrated blind gated run remains null (+0.0014 cosine, p=.74). The model file
`gate_model.joblib` and the thresholds in `gate_meta.json` are the frozen artifacts; they are fit on
dev and applied to eval unchanged.

## Conclusion — the general gate

A single gate trained across all routing signals and both protocols learns a general benefit signal
(grouped dev OOF AUC 0.699 [0.676, 0.721]) and transfers to eval above chance for every configuration.
It is risk control, not gain: it recovers 0.54–0.56 of the ceiling where harm is concentrated
(uncalibrated ticket-only feedback) and is neutral once calibration has removed that concentration.
The open/closed decomposition explains why, and expected-value (magnitude) modelling — not sign
classification — is the right layer when a per-ticket action must be chosen: it closes smaller, more
harmful subsets and turns action selection positive (+0.0116 [+0.0034, +0.0205]).